# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vishalseries7030/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

### 1. Unit of analysis + time window

The primary daily performance table has one row per report date, pseudonymized client, and pseudonymized content item. The inspected sample contains 11,694,072 rows covering 1–30 June 2026.

The content dimension contains 519,606 rows, with one row per pseudonymized content item. The query-performance table contains 2,414,248 rows, one row per pseudonymized client, content item, and query hash, for a fixed 90-day window from 2 April to 30 June 2026.

These tables have different grains and must not be joined without checking key uniqueness and join cardinality. The June daily sample will be used for initial exploration; it is not sufficient by itself to establish future performance or causal impact.

In [ ]:

# ML-04: Inspect the current notebook environment
import os
from pathlib import Path

print("Current directory:", os.getcwd())
print("\nAvailable files in current directory:")
for path in list(Path(".").iterdir())[:30]:
    print(path)


Current directory: /content

Available files in current directory:
.config
sample_data


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

### 2. Fields: feature / label / context / excluded

**Candidate features:** content metadata such as `content_type`, `search_volume`, `competition`, `main_intent`, `word_count`, `char_count`, `backlinks`, and `category_count`, subject to missingness and availability checks.

**Candidate outcome measures:** `gsc_clicks`, `gsc_impressions`, and `gsc_avg_position`. A prediction target and prediction horizon have not yet been finalized; same-period performance must not be used as a feature to predict that same period.

**Context fields:** `report_date`, `month`, `client_has_gsc`, `client_has_ga4`, `gsc_data_available`, and `ga4_data_available`.

**Identifiers and exclusions:** hashed client, content, keyword, URL, and query identifiers are for controlled joins and grouping, not direct predictive features. Raw or identifying client information, domains, URLs, private queries, and credentials must not appear in public outputs.

Missing performance values must be distinguished from genuine zero activity using the relevant availability flags. Join cardinality, feature timestamps, and possible leakage must be checked before modeling.

In [ ]:

import os

print("HF token available:", bool(os.environ.get("HF_TOKEN")))
print("Hugging Face token available:", bool(os.environ.get("HUGGINGFACEHUB_API_TOKEN")))


HF token available: False
Hugging Face token available: False


In [ ]:

!pip -q install huggingface_hub
from huggingface_hub import HfApi

api = HfApi()
try:
    datasets = api.list_datasets(search="FlyRank", limit=20)
    for dataset in datasets:
        print(dataset.id)
except Exception as e:
    print(type(e).__name__, str(e)[:500])


FlyRank/internship-warehouse
FlyRank/internship-starter
FlyRank/internship-lanes


In [ ]:

from huggingface_hub import HfApi

api = HfApi()
repo_id = "FlyRank/internship-warehouse"

try:
    info = api.dataset_info(repo_id)
    print("Dataset:", info.id)
    print("\nFiles:")
    for file in api.list_repo_files(repo_id, repo_type="dataset"):
        print(file)
except Exception as e:
    print(type(e).__name__, str(e)[:1000])


Dataset: FlyRank/internship-warehouse

Files:
.gitattributes
README.md
dim_clients.parquet
dim_content.parquet
fact_content_daily_performance/month=2025-01/data_0.parquet
fact_content_daily_performance/month=2025-02/data_0.parquet
fact_content_daily_performance/month=2025-03/data_0.parquet
fact_content_daily_performance/month=2025-04/data_0.parquet
fact_content_daily_performance/month=2025-05/data_0.parquet
fact_content_daily_performance/month=2025-06/data_0.parquet
fact_content_daily_performance/month=2025-07/data_0.parquet
fact_content_daily_performance/month=2025-08/data_0.parquet
fact_content_daily_performance/month=2025-09/data_0.parquet
fact_content_daily_performance/month=2025-10/data_0.parquet
fact_content_daily_performance/month=2025-11/data_0.parquet
fact_content_daily_performance/month=2025-12/data_0.parquet
fact_content_daily_performance/month=2026-01/data_0.parquet
fact_content_daily_performance/month=2026-02/data_0.parquet
fact_content_daily_performance/month=2026-03/data

In [ ]:

!pip -q install pandas pyarrow huggingface_hub

import pandas as pd
from huggingface_hub import hf_hub_download

repo = "FlyRank/internship-warehouse"

for name in [
    "README.md",
    "dim_content.parquet",
    "fact_content_daily_performance_sample.parquet",
    "fact_content_query_90d.parquet",
]:
    print("\n--- FILE:", name, "---")
    try:
        path = hf_hub_download(
            repo_id=repo,
            repo_type="dataset",
            filename=name
        )
        if name.endswith(".parquet"):
            df = pd.read_parquet(path)
            print("Shape:", df.shape)
            print("Columns:", df.columns.tolist())
            print("Data types:\n", df.dtypes.to_string())
        else:
            with open(path, encoding="utf-8") as f:
                print(f.read()[:3000])
    except Exception as e:
        print("ERROR:", str(e)[:500])



--- FILE: README.md ---
---
license: other
language:
- en
tags:
- seo
- content-performance
- data-warehouse
- tabular
- education
- flyrank-internship
pretty_name: FlyRank Internship — Warehouse Star Schema (Pseudonymized, Gated)
size_categories:
- 10M<n<100M
extra_gated_prompt: >-
  By requesting access you agree to the FlyRank Internship Data Use Terms:
  anonymized research and education use only; no attempt to re-identify clients,
  domains, queries, keywords, or content; no redistribution of the raw data; and
  no client-identifying data in any public output (case study, repo, chart, or demo).
extra_gated_fields:
  Name: text
  Email: text
  Affiliation or cohort: text
  I agree to the FlyRank data-use terms: checkbox
configs:
- config_name: dim_clients
  data_files: dim_clients.parquet
- config_name: dim_content
  data_files: dim_content.parquet
- config_name: fact_content_daily_performance
  data_files: fact_content_daily_performance/**/*.parquet
- config_name: fact_content_qu

In [ ]:

from huggingface_hub import HfApi

api = HfApi()

try:
    info = api.dataset_info("FlyRank/internship-warehouse")
    print("Dataset metadata accessible:", info.id)

    files = api.list_repo_files(
        "FlyRank/internship-warehouse",
        repo_type="dataset"
    )
    print("Warehouse files visible:", len(files))
except Exception as e:
    print(type(e).__name__, str(e)[:500])


Dataset metadata accessible: FlyRank/internship-warehouse
Warehouse files visible: 24


In [1]:

from huggingface_hub import login

login()
print("Hugging Face login successful!")

Hugging Face login successful!


In [ ]:

import pandas as pd
from huggingface_hub import hf_hub_download

repo = "FlyRank/internship-warehouse"
files = [
    "dim_content.parquet",
    "fact_content_daily_performance_sample.parquet",
    "fact_content_query_90d.parquet",
]

for name in files:
    print("\n" + "=" * 50)
    print("FILE:", name)
    try:
        path = hf_hub_download(
            repo_id=repo,
            repo_type="dataset",
            filename=name
        )
        df = pd.read_parquet(path)
        print("Shape:", df.shape)
        print("Columns:", df.columns.tolist())
        print("\nData types:\n", df.dtypes.to_string())
    except Exception as e:
        print("ERROR:", type(e).__name__, str(e)[:400])



FILE: dim_content.parquet


dim_content.parquet: reconstructing file:   0%|          |  0.00B / 19.6MB            

dim_content.parquet: downloading bytes:           |  0.00B            

Shape: (519606, 26)
Columns: ['client_hash_id', 'content_hash_id', 'keyword_hash_id', 'url_hash_id', 'keyword_char_count', 'keyword_token_count', 'url_char_count', 'content_created_date', 'content_updated_date', 'content_type', 'search_volume', 'competition', 'competition_level', 'cpc', 'main_intent', 'backlinks', 'category_count', 'keyword_created_date', 'provider_used', 'model_used', 'char_count', 'word_count', 'last_optimized_date', 'optimization_eligible_date', 'is_published', 'is_deleted']

Data types:
 client_hash_id                 object
content_hash_id                object
keyword_hash_id                object
url_hash_id                    object
keyword_char_count              int64
keyword_token_count             int64
url_char_count                  int64
content_created_date           object
content_updated_date           object
content_type                   object
search_volume                 float64
competition                   float64
competition_level             

fact_content_daily_performance_sample.pa(…): reconstructing file:   0%|          |  0.00B /  145MB            

fact_content_daily_performance_sample.pa(…): downloading bytes:           |  0.00B            

Shape: (11694072, 31)
Columns: ['report_date', 'client_hash_id', 'content_hash_id', 'client_has_gsc', 'client_has_ga4', 'gsc_data_available', 'ga4_data_available', 'gsc_impressions', 'gsc_clicks', 'gsc_sum_position', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions', 'ga4_users', 'ga4_engaged_sessions', 'ga4_total_engagement_sec', 'sessions_organic', 'sessions_direct', 'sessions_referral', 'sessions_social', 'sessions_paid', 'sessions_ai', 'ai_chatgpt', 'ai_perplexity', 'ai_gemini', 'ai_copilot', 'ai_claude', 'ai_meta', 'ai_other', 'scroll_events', 'month']

Data types:
 report_date                  object
client_hash_id               object
content_hash_id              object
client_has_gsc                 bool
client_has_ga4                 bool
gsc_data_available             bool
ga4_data_available           object
gsc_impressions               int64
gsc_clicks                    int64
gsc_sum_position            float64
gsc_avg_position            float64
ga4_pageviews           

fact_content_query_90d.parquet: reconstructing file:   0%|          |  0.00B / 60.7MB            

fact_content_query_90d.parquet: downloading bytes:           |  0.00B            

Shape: (2414248, 21)
Columns: ['client_hash_id', 'content_hash_id', 'query_hash_id', 'query_char_count', 'query_token_count', 'window_start', 'window_end', 'impressions_90d', 'clicks_90d', 'impressions_last30', 'clicks_last30', 'impressions_prev30', 'clicks_prev30', 'avg_position_90d', 'avg_position_last30', 'avg_position_prev30', 'content_total_impressions_90d', 'content_visible_query_count', 'rare_query_count', 'rare_impressions_share', 'anonymized_impressions_share']

Data types:
 client_hash_id                    object
content_hash_id                   object
query_hash_id                     object
query_char_count                   int64
query_token_count                  int64
window_start                      object
window_end                        object
impressions_90d                    int64
clicks_90d                         int64
impressions_last30                 int64
clicks_last30                      int64
impressions_prev30                 int64
clicks_prev30      

In [ ]:

import pandas as pd
from huggingface_hub import hf_hub_download

repo = "FlyRank/internship-warehouse"

# 1. Daily performance: read only the columns needed for audit
daily_path = hf_hub_download(
    repo_id=repo,
    repo_type="dataset",
    filename="fact_content_daily_performance_sample.parquet"
)

daily = pd.read_parquet(
    daily_path,
    columns=[
        "report_date", "gsc_data_available", "ga4_data_available",
        "gsc_impressions", "gsc_clicks", "gsc_avg_position",
        "ga4_pageviews", "ga4_sessions"
    ]
)

daily["report_date"] = pd.to_datetime(daily["report_date"], errors="coerce")

print("DAILY PERFORMANCE AUDIT")
print("Rows:", len(daily))
print("Date range:", daily["report_date"].min(), "to", daily["report_date"].max())
print("\nMissing-value percentage:")
print((daily.isna().mean() * 100).round(2).to_string())
print("\nData availability:")
print("GSC:\n", daily["gsc_data_available"].value_counts(dropna=False).to_string())
print("GA4:\n", daily["ga4_data_available"].value_counts(dropna=False).to_string())

# 2. Query table: verify its reporting window
query_path = hf_hub_download(
    repo_id=repo,
    repo_type="dataset",
    filename="fact_content_query_90d.parquet"
)

query = pd.read_parquet(
    query_path,
    columns=["window_start", "window_end", "impressions_90d", "clicks_90d"]
)

print("\nQUERY PERFORMANCE AUDIT")
print("Rows:", len(query))
print("Window start range:",
      query["window_start"].min(), "to", query["window_start"].max())
print("Window end range:",
      query["window_end"].min(), "to", query["window_end"].max())
print("Missing-value percentage:")
print((query.isna().mean() * 100).round(2).to_string())

print("\nAudit complete.")


DAILY PERFORMANCE AUDIT
Rows: 11694072
Date range: 2026-06-01 00:00:00 to 2026-06-30 00:00:00

Missing-value percentage:
report_date            0.00
gsc_data_available     0.00
ga4_data_available    20.50
gsc_impressions        0.00
gsc_clicks             0.00
gsc_avg_position      66.83
ga4_pageviews         20.50
ga4_sessions          20.50

Data availability:
GSC:
 gsc_data_available
False    7815135
True     3878937
GA4:
 ga4_data_available
False    8651918
None     2397428
True      644726

QUERY PERFORMANCE AUDIT
Rows: 2414248
Window start range: 2026-04-02 to 2026-04-02
Window end range: 2026-06-30 to 2026-06-30
Missing-value percentage:
window_start       0.0
window_end         0.0
impressions_90d    0.0
clicks_90d         0.0

Audit complete.


In [ ]:

!pip -q install pandas pyarrow huggingface_hub

import pandas as pd
from huggingface_hub import hf_hub_download

repo_id = "FlyRank/internship-warehouse"
files = [
    "README.md",
    "dim_content.parquet",
    "fact_content_daily_performance_sample.parquet",
    "fact_content_query_90d.parquet",
]

for file in files:
    print("\n" + "=" * 60)
    print("FILE:", file)

    try:
        path = hf_hub_download(
            repo_id=repo_id,
            repo_type="dataset",
            filename=file
        )

        if file.endswith(".parquet"):
            df = pd.read_parquet(path)
            print("Shape:", df.shape)
            print("Columns:", df.columns.tolist())
            print("Data types:")
            print(df.dtypes.to_string())
            print("Missing values:")
            print(df.isna().sum().to_string())
        else:
            with open(path, "r", encoding="utf-8") as f:
                print(f.read()[:5000])

    except Exception as e:
        print(type(e).__name__, str(e)[:500])



FILE: README.md


README.md:   0%|          | 0.00/3.04k [00:00<?, ?B/s]

---
license: other
language:
- en
tags:
- seo
- content-performance
- data-warehouse
- tabular
- education
- flyrank-internship
pretty_name: FlyRank Internship — Warehouse Star Schema (Pseudonymized, Gated)
size_categories:
- 10M<n<100M
extra_gated_prompt: >-
  By requesting access you agree to the FlyRank Internship Data Use Terms:
  anonymized research and education use only; no attempt to re-identify clients,
  domains, queries, keywords, or content; no redistribution of the raw data; and
  no client-identifying data in any public output (case study, repo, chart, or demo).
extra_gated_fields:
  Name: text
  Email: text
  Affiliation or cohort: text
  I agree to the FlyRank data-use terms: checkbox
configs:
- config_name: dim_clients
  data_files: dim_clients.parquet
- config_name: dim_content
  data_files: dim_content.parquet
- config_name: fact_content_daily_performance
  data_files: fact_content_daily_performance/**/*.parquet
- config_name: fact_content_query_90d
  data_files: fac

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

### 3. Verify it with queries

The June 2026 daily performance sample contains 11,694,072 rows across 30 report dates. The expected daily key (`report_date`, `client_hash_id`, `content_hash_id`) has 6,390 duplicate rows, so the key is not unique in the raw sample. These duplicates require investigation before aggregation or modeling.

The content dimension contains 519,606 rows and 519,606 unique (`client_hash_id`, `content_hash_id`) pairs. All 409,205 distinct client-content pairs observed in the daily sample matched the content dimension; no unmatched pairs were found.

The join check supports a many-to-one join from distinct daily client-content pairs to the content dimension, but does not resolve duplicate daily keys. Duplicate patterns and their effect on metrics must be investigated before modeling.

The June 2026 sample contains 6,390 duplicate daily keys, representing 12,780 rows across duplicate-key groups. Inspection confirmed that both rows in every duplicate group are identical across all columns. These exact duplicate records should be removed in the analysis copy only, while preserving the original source data. The cleaned analysis data must be revalidated to confirm that the daily key (report_date, client_hash_id, content_hash_id) is unique.

In [ ]:

# Inspect duplicate daily keys without dropping any rows
key_cols = ["report_date", "client_hash_id", "content_hash_id"]

dup = daily_keys[
    daily_keys.duplicated(key_cols, keep=False)
].sort_values(key_cols)

print("Rows belonging to duplicate-key groups:", len(dup))
print("\nSample duplicate groups:")
print(dup.head(12).to_string(index=False))

print("\nNumber of duplicated key groups:",
      dup.drop_duplicates(key_cols).shape[0])


Rows belonging to duplicate-key groups: 12780

Sample duplicate groups:
report_date          client_hash_id          content_hash_id
 2026-06-13 client_06d356715a8ff3b6 content_01dd034a486dd411
 2026-06-13 client_06d356715a8ff3b6 content_01dd034a486dd411
 2026-06-13 client_06d356715a8ff3b6 content_03a8b5950a52518a
 2026-06-13 client_06d356715a8ff3b6 content_03a8b5950a52518a
 2026-06-13 client_06d356715a8ff3b6 content_1fa28219b658d722
 2026-06-13 client_06d356715a8ff3b6 content_1fa28219b658d722
 2026-06-13 client_06d356715a8ff3b6 content_27a329abc15773a7
 2026-06-13 client_06d356715a8ff3b6 content_27a329abc15773a7
 2026-06-13 client_06d356715a8ff3b6 content_2b4f82281f289e87
 2026-06-13 client_06d356715a8ff3b6 content_2b4f82281f289e87
 2026-06-13 client_06d356715a8ff3b6 content_2dcd919936137680
 2026-06-13 client_06d356715a8ff3b6 content_2dcd919936137680

Number of duplicated key groups: 6390


In [ ]:
import pandas as pd

key_cols = ["report_date", "client_hash_id", "content_hash_id"]

# Full rows for the duplicate keys
daily = pd.read_parquet(daily_path)

dup_keys = daily[daily.duplicated(key_cols, keep=False)]

# Check whether duplicate rows are identical across all columns
duplicate_groups = (
    dup_keys.groupby(key_cols, dropna=False)
    .size()
    .reset_index(name="rows_in_group")
)

print("Duplicate groups:", len(duplicate_groups))
print("\nRows per duplicate group:")
print(duplicate_groups["rows_in_group"].value_counts().sort_index())

print("\nAre all duplicate rows identical across every column?")
print(
    dup_keys.groupby(key_cols, dropna=False)
    .apply(lambda group: group.drop(columns=key_cols).nunique(dropna=False).max() == 1)
    .value_counts()
)

Duplicate groups: 6390

Rows per duplicate group:
rows_in_group
2    6390
Name: count, dtype: int64

Are all duplicate rows identical across every column?
True    6390
Name: count, dtype: int64


/tmp/ipykernel_3539/307224453.py:24: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda group: group.drop(columns=key_cols).nunique(dropna=False).max() == 1)


In [ ]:
# Preserve the original dataset
daily_raw = daily.copy()

# Remove only exact duplicate rows
daily_clean = daily_raw.drop_duplicates().copy()

print("Original rows:", len(daily_raw))
print("Rows after exact deduplication:", len(daily_clean))
print("Rows removed:", len(daily_raw) - len(daily_clean))

# Verify the expected daily key is now unique
key_cols = ["report_date", "client_hash_id", "content_hash_id"]

print(
    "Remaining duplicate key rows:",
    daily_clean.duplicated(key_cols, keep=False).sum()
)

NameError: name 'daily' is not defined

In [ ]:
import pandas as pd

key_cols = ["report_date", "client_hash_id", "content_hash_id"]

daily = pd.read_parquet(daily_path)

daily_raw = daily.copy()
daily_clean = daily_raw.drop_duplicates().copy()

print("Original rows:", len(daily_raw))
print("Rows after cleaning:", len(daily_clean))
print("Exact duplicate rows removed:", len(daily_raw) - len(daily_clean))
print(
    "Remaining duplicate key rows:",
    daily_clean.duplicated(key_cols, keep=False).sum()
)

NameError: name 'daily_path' is not defined

In [ ]:
import os

for root, dirs, files in os.walk("/"):
    if "fact_content_daily_performance_sample.parquet" in files:
        print(os.path.join(root, "fact_content_daily_performance_sample.parquet"))
        break

/root/.cache/huggingface/hub/datasets--FlyRank--internship-warehouse/snapshots/50cbf7c3909d07be4d1b5906b4d09e882e5acbf2/fact_content_daily_performance_sample.parquet


In [ ]:
import pandas as pd

daily_path = "/root/.cache/huggingface/hub/datasets--FlyRank--internship-warehouse/snapshots/50cbf7c3909d07be4d1b5906b4d09e882e5acbf2/fact_content_daily_performance_sample.parquet"

daily = pd.read_parquet(daily_path)

key_cols = ["report_date", "client_hash_id", "content_hash_id"]

daily_raw = daily.copy()
daily_clean = daily_raw.drop_duplicates().copy()

print("Original rows:", len(daily_raw))
print("Rows after cleaning:", len(daily_clean))
print("Exact duplicate rows removed:", len(daily_raw) - len(daily_clean))
print(
    "Remaining duplicate key rows:",
    daily_clean.duplicated(key_cols, keep=False).sum()
)

In [ ]:
import os

daily_path = "/root/.cache/huggingface/hub/datasets--FlyRank--internship-warehouse/snapshots/50cbf7c3909d07be4d1b5906b4d09e882e5acbf2/fact_content_daily_performance_sample.parquet"

print("File exists:", os.path.exists(daily_path))

if os.path.exists(daily_path):
    print("File size (GB):", round(os.path.getsize(daily_path) / (1024**3), 2))

File exists: True
File size (GB): 0.14


In [ ]:
import pyarrow.parquet as pq

parquet_file = pq.ParquetFile(daily_path)

print("Total row groups:", parquet_file.num_row_groups)
print("Total rows:", parquet_file.metadata.num_rows)
print("Columns:", parquet_file.schema.names[:8])

Total row groups: 98
Total rows: 11694072
Columns: ['report_date', 'client_hash_id', 'content_hash_id', 'client_has_gsc', 'client_has_ga4', 'gsc_data_available', 'ga4_data_available', 'gsc_impressions']


In [ ]:
import pyarrow.parquet as pq
import pandas as pd

pf = pq.ParquetFile(daily_path)

key_cols = ["report_date", "client_hash_id", "content_hash_id"]
key_frames = []

for batch in pf.iter_batches(
    batch_size=100_000,
    columns=key_cols
):
    key_frames.append(batch.to_pandas())

keys = pd.concat(key_frames, ignore_index=True)

print("Total key rows:", len(keys))
print("Duplicate key rows:", keys.duplicated(key_cols, keep=False).sum())
print("Duplicate key groups:", keys.duplicated(key_cols).sum())

Total key rows: 11694072
Duplicate key rows: 12780
Duplicate key groups: 6390


In [ ]:
# Check the duplicate keys without loading the full dataset
duplicate_keys = keys.loc[
    keys.duplicated(
        ["report_date", "client_hash_id", "content_hash_id"],
        keep=False
    ),
    ["report_date", "client_hash_id", "content_hash_id"]
].drop_duplicates()

print("Unique duplicate keys:", len(duplicate_keys))
print("Expected duplicate groups:", 6390)
print("Counts match:", len(duplicate_keys) == 6390)

Unique duplicate keys: 6390
Expected duplicate groups: 6390
Counts match: True


In [ ]:

# ML-04: Verify grain, key uniqueness, and join safety
from huggingface_hub import hf_hub_download
import pandas as pd

repo = "FlyRank/internship-warehouse"

# Load only join keys from the June daily sample and content dimension
daily_path = hf_hub_download(
    repo_id=repo,
    repo_type="dataset",
    filename="fact_content_daily_performance_sample.parquet"
)
content_path = hf_hub_download(
    repo_id=repo,
    repo_type="dataset",
    filename="dim_content.parquet"
)

daily_keys = pd.read_parquet(
    daily_path,
    columns=["report_date", "client_hash_id", "content_hash_id"]
)
content_keys = pd.read_parquet(
    content_path,
    columns=["client_hash_id", "content_hash_id"]
)

print("DAILY TABLE")
print("Rows:", len(daily_keys))
print("Unique report dates:", daily_keys["report_date"].nunique())
print("Unique client-content-date keys:",
      daily_keys.drop_duplicates(
          ["report_date", "client_hash_id", "content_hash_id"]
      ).shape[0])
print("Duplicate daily keys:",
      daily_keys.duplicated(
          ["report_date", "client_hash_id", "content_hash_id"]
      ).sum())

print("\nCONTENT DIMENSION")
print("Rows:", len(content_keys))
print("Unique client-content keys:",
      content_keys.drop_duplicates(
          ["client_hash_id", "content_hash_id"]
      ).shape[0])
print("Duplicate content keys:",
      content_keys.duplicated(
          ["client_hash_id", "content_hash_id"]
      ).sum())

print("\nDAILY CONTENT KEYS MISSING FROM DIMENSION")
daily_pairs = daily_keys[["client_hash_id", "content_hash_id"]].drop_duplicates()
content_pairs = content_keys.drop_duplicates()
check = daily_pairs.merge(
    content_pairs,
    on=["client_hash_id", "content_hash_id"],
    how="left",
    indicator=True,
    validate="many_to_one"
)
print("Distinct daily pairs:", len(daily_pairs))
print("Unmatched pairs:", (check["_merge"] == "left_only").sum())

print("\nVerification complete.")


DAILY TABLE
Rows: 11694072
Unique report dates: 30
Unique client-content-date keys: 11687682
Duplicate daily keys: 6390

CONTENT DIMENSION
Rows: 519606
Unique client-content keys: 519606
Duplicate content keys: 0

DAILY CONTENT KEYS MISSING FROM DIMENSION
Distinct daily pairs: 409205
Unmatched pairs: 0

Verification complete.


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

The inspected daily performance sample covers only June 2026, so it cannot establish long-term trends, seasonality, or future performance. Data availability is uneven: GSC metrics are unavailable for many rows, and GA4 availability is also incomplete. Missing metrics must not automatically be interpreted as zero activity. The query-performance table covers a fixed 90-day window from 2 April to 30 June 2026, which overlaps the June daily sample and has a different unit of analysis. Combining these tables requires careful date alignment, join-cardinality checks, and leakage prevention. The observational data can support descriptive analysis and cautious decision-support recommendations, but cannot by itself establish that a particular change caused an improvement in search performance.

In [ ]:
print("Daily sample date range:", keys["report_date"].min(), "to", keys["report_date"].max())
print("Total daily rows:", len(keys))
print("Duplicate key groups:", len(duplicate_keys))
print("Duplicate key rows:", keys.duplicated(
    ["report_date", "client_hash_id", "content_hash_id"], keep=False
).sum())

Daily sample date range: 2026-06-01 to 2026-06-30
Total daily rows: 11694072
Duplicate key groups: 6390
Duplicate key rows: 12780


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.